# M3 — corrección v6: diagnóstico, no evaluación final
Usa una sesión nueva de Colab con GPU T4. Sube únicamente `contractrisk_m3_correccion_v6_codigo.zip`. Se reutilizan los contextos de desarrollo v5 para aislar cambios de generación; el agente vuelve a decidir y ejecutar la herramienta. Se guardan 32 respuestas y 48 juicios sintéticos. No se ejecuta el gold ni RAGAS. La latencia no es comparable con recuperación real. Conserva los resultados aunque fallen; el éxito técnico no certifica calidad.

In [ ]:
from google.colab import files
from pathlib import Path
import tempfile, zipfile, os, sys, subprocess, shutil
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Sube un solo ZIP de código v6')
ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_m3_v6_', dir='/content'))
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    for name in archive.namelist():
        if not (ROOT / name).resolve().is_relative_to(ROOT): raise ValueError('Ruta ZIP insegura')
    archive.extractall(ROOT)
os.chdir(ROOT)
assert (ROOT/'m3/run_correction_v6.py').exists()
print(ROOT)


In [ ]:
def logged(command, destination):
    with open(destination, 'w', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    if code: raise subprocess.CalledProcessError(code, command)
logged([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], ROOT/'installation.log')
import torch
assert torch.cuda.is_available(), 'Activa GPU T4'
print('GPU:', torch.cuda.get_device_name(0))
logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], ROOT/'tests.log')


In [ ]:
# No repetir con la misma carpeta: conserva cualquier evidencia anterior.
logged([sys.executable, '-u', 'm3/run_correction_v6.py', '--output', 'v6_resultados'], ROOT/'execution.log')


In [ ]:
# Ejecuta también esta celda si falla la anterior.
OUT = ROOT/'v6_resultados'
OUT.mkdir(exist_ok=True)
for filename in ['installation.log','tests.log','execution.log']:
    if (ROOT/filename).exists(): shutil.copy2(ROOT/filename, OUT/filename)
import json
if (OUT/'metadata.json').exists():
    print((OUT/'metadata.json').read_text(encoding='utf-8'))
if (OUT/'calibration_summary.json').exists():
    summary=json.loads((OUT/'calibration_summary.json').read_text(encoding='utf-8'))
    print(json.dumps({k:{a:b for a,b in v.items() if a!='pairs'} for k,v in summary.items()}, indent=2))
path=shutil.make_archive(str(ROOT/'contractrisk_m3_correccion_v6_resultados'), 'zip', OUT)
files.download(path)
print('Devuelve este ZIP y descarga el notebook con sus salidas.')
